## STADIOchoice Churn Prediction

### Objective and problem definition

Objective: Test whether subscriber/account information and behaviour available at a monthly index date can rank subscribers by the probability of a qualifying churn event in the following **30 days**.

The supplied data dictionary defines churn as subscriber-initiated cancellation without reactivation within 30 days, and the supplied `churn_indicator` is the authoritative outcome flag. A 30-day prediction horizon is therefore distinct from the 30-day reactivation rule.

Observation grain: One eligible subscriber per month-end index date. A subscriber may appear in multiple periods because the intended use is forecasting future behaviour for existing subscribers. Features use only records dated on or before the index date; the label uses the following 30 days. The final outcome table and final subscription status are never predictors.

### Environment Setup and Imports

In [2]:

from pathlib import Path
import sys
import platform
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone



SEED = 42
HORIZON_DAYS = 30
LOOKBACK_DAYS = 90

CURRENT_DIR = Path.cwd().resolve()

possible_roots = [CURRENT_DIR, *CURRENT_DIR.parents]

PROJECT_ROOT = next(
    (
        root
        for root in possible_roots
        if (root / "data" / "stadiochoice").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project data directory.\n"
        f"Current working directory: {CURRENT_DIR}\n"
        "Expected: <project-root>/data/stadiochoice/"
    )

DATA_DIR = PROJECT_ROOT / "data" / "stadiochoice"

FILES = {
    "account": "2_subscriber_account.csv",
    "subscription": "3_subscription.csv",
    "viewing": "4_viewing_behaviour.csv",
    "engagement": "5_engagement_behaviour.csv",
    "content": "5b_content_reference.csv",
    "billing": "6_billing_payment_history.csv",
    "outcome": "7_cancellation_retention.csv",
}

print(
    f"Python: {platform.python_version()} | "
    f"pandas: {pd.__version__} | "
    f"NumPy: {np.__version__}"
)
print(f"Project root: {PROJECT_ROOT}")
print(f"Input directory: {DATA_DIR}")

missing_files = [
    filename
    for filename in FILES.values()
    if not (DATA_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files in {DATA_DIR}:\n"
        + "\n".join(missing_files)
    )

print(f"All {len(FILES)} required CSV files are present.")

Python: 3.13.7 | pandas: 3.0.5 | NumPy: 2.5.2
Project root: C:\Users\sltha\Documents\STADIO\CAP182\capstone-project
Input directory: C:\Users\sltha\Documents\STADIO\CAP182\capstone-project\data\stadiochoice
All 7 required CSV files are present.


### Data inventory and loading

The README describes seven operational tables and their intended grains. We load them without silently dropping records, parse timestamp fields explicitly, and compare actual shapes with the documented counts. The content lookup enriches viewing records, but the churn modelling table will be built at subscriber-month grain.

In [3]:
tables = {
    name: pd.read_csv(DATA_DIR / filename)
    for name, filename in FILES.items()
}

date_cols = {
    "account": [
        "account_creation_date",
        "acquisition_date",
    ],
    "subscription": [
        "subscription_start_date",
        "subscription_end_date",
    ],
    "viewing": [
        "viewing_timestamp",
    ],
    "engagement": [
        "event_timestamp",
    ],
    "billing": [
        "billing_date",
    ],
    "outcome": [
        "cancellation_date",
        "reactivation_date",
        "retention_interaction_date",
    ],
}

for table, columns in date_cols.items():
    for column in columns:
        tables[table][column] = pd.to_datetime(
            tables[table][column],
            errors="coerce",
        )


table_grains = {
    "account": "one row per subscriber",
    "subscription": "one row per subscription",
    "viewing": "one row per viewing session",
    "engagement": "one row per engagement event",
    "content": "one row per content item",
    "billing": "one row per billing event",
    "outcome": "one row per cancellation/retention event",
}

inventory = pd.DataFrame([
    {
        "table": name,
        "file": FILES[name],
        "rows": len(df),
        "columns": df.shape[1],
        "subscriber_ids": (
            df["subscriber_id"].nunique()
            if "subscriber_id" in df.columns
            else np.nan
        ),
        "grain_from_documentation": table_grains[name],
    }
    for name, df in tables.items()
])

display(inventory)


for name, df in tables.items():
    print(f"\n{name}: {df.shape}")

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (
            100 * df.isna().mean()
        ).round(2),
    })

    display(summary)


,table,file,rows,columns,subscriber_ids,grain_from_documentation
0,account,2_subscriber_account.csv,2110,8,2104.0,one row per subscriber
1,subscription,3_subscription.csv,2104,9,2104.0,one row per subscription
2,viewing,4_viewing_behaviour.csv,8939,11,2104.0,one row per viewing session
3,engagement,5_engagement_behaviour.csv,28118,7,2104.0,one row per engagement event
4,content,5b_content_reference.csv,880,3,NaN,one row per content item
5,billing,6_billing_payment_history.csv,32114,9,2104.0,one row per billing event
6,outcome,7_cancellation_retention.csv,539,11,539.0,one row per cancellation/retention event



account: (2110, 8)


,dtype,missing,missing_pct
subscriber_id,str,0,0.00
household_id,str,0,0.00
account_creation_date,datetime64[us],0,0.00
acquisition_date,datetime64[us],0,0.00
acquisition_channel,str,0,0.00
acquisition_campaign,str,278,13.18
customer_segment,str,0,0.00
province,str,85,4.03



subscription: (2104, 9)


,dtype,missing,missing_pct
subscriber_id,str,0,0.00
subscription_id,str,0,0.00
subscription_plan,str,0,0.00
subscription_start_date,datetime64[us],0,0.00
subscription_end_date,datetime64[us],1727,82.08
subscription_status,str,0,0.00
monthly_fee,float64,0,0.00
discount_amount,float64,0,0.00
promotion_code,str,966,45.91


...

,dtype,missing,missing_pct
content_id,str,0,0.0
title,str,0,0.0
category,str,0,0.0



billing: (32114, 9)


,dtype,missing,missing_pct
subscriber_id,str,0,0.0
billing_date,datetime64[us],0,0.0
amount_charged,float64,0,0.0
payment_status,str,0,0.0
payment_method,str,0,0.0
payment_failure_indicator,bool,0,0.0
payment_failure_count,int64,0,0.0
refund_indicator,bool,0,0.0
refund_amount,float64,0,0.0



outcome: (539, 11)


,dtype,missing,missing_pct
subscriber_id,str,0,0.00
cancellation_date,datetime64[us],0,0.00
cancellation_reason,str,0,0.00
cancellation_channel,str,0,0.00
churn_indicator,bool,0,0.00
reactivation_indicator,bool,0,0.00
reactivation_date,datetime64[us],497,92.21
retention_interaction_date,datetime64[us],134,24.86
retention_offer,str,134,24.86
offer_accepted,object,134,24.86
